In [37]:
# ============================================================
# CELL 1 — IMPORTS E CONFIGURAÇÃO
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

Device: cuda
NVIDIA RTX 6000 Ada Generation


In [38]:
# ============================================================
# CELL 2 — PATHS E HIPERPARÂMETROS
# ============================================================

DATA_ROOT    = Path("/mounts/mecd-ap-g5/data")
DATASET_ROOT = DATA_ROOT / "MIQR-CC-Dataset"
METADATA_PATH = DATASET_ROOT / "metadata.csv"

RESULTS_DIR = Path("./results_efficientnet")
RESULTS_DIR.mkdir(exist_ok=True)

MODELS_DIR = Path("./models_efficientnet")
MODELS_DIR.mkdir(exist_ok=True)

# Hiperparâmetros
IMG_SIZE   = 224      # EfficientNet-B3 aceita 300, mas 224 é mais rápido e suficiente
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-4
LR_BACKBONE = 1e-5   # learning rate mais baixo para as camadas pré-treinadas
WEIGHT_DECAY = 1e-4

print(METADATA_PATH)

/mounts/mecd-ap-g5/data/MIQR-CC-Dataset/metadata.csv


In [39]:
# ============================================================
# CELL 3 — CARREGAR E LIMPAR METADADOS
# ============================================================

df = pd.read_csv(METADATA_PATH)
print("Shape original:", df.shape)
print("Colunas:", df.columns.tolist())
print("\nValores únicos em 'Keep':", df["Keep"].unique())
print("\nDistribuição de classes (Label):")
print(df["Label"].value_counts())

Shape original: (19317, 12)
Colunas: ['raw_image_path', 'processed_image_path', 'patient_id', 'image_type', 'sex', 'birth_date', 'exam_date', 'exam_time', 'age', 'equipment_model', 'Label', 'Keep']

Valores únicos em 'Keep': ['Keep' 'Discard']

Distribuição de classes (Label):
Label
Unlabelled             13798
Lithiasis               2651
Malignant Stricture     1281
Normal                  1012
Biliary Leaks            361
Benign Stricture         214
Name: count, dtype: int64


In [40]:
# ============================================================
# CELL 4 — FILTRAR, MAPEAR CLASSES E CONSTRUIR PATHS
# ============================================================

# ATENÇÃO: o valor na coluna é "Keep" com K maiúsculo
df = df[df["Keep"] == "Keep"].copy()
print("Após filtro Keep:", df.shape)

LABEL_MAP = {
    "Benign Stricture":   "Stricture",
    "Malignant Stricture": "Stricture",
    "Lithiasis":          "Lithiasis",
    "Normal":             "Normal",
    "Biliary Leaks":      "Biliary Leaks",
}

df["label_4class"] = df["Label"].map(LABEL_MAP)
df = df[df["label_4class"].notna()].copy()  # remove "Unlabelled"

class_names = ["Biliary Leaks", "Lithiasis", "Normal", "Stricture"]
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

df["target"] = df["label_4class"].map(class_to_idx)

# Construir path completo
df["image_path"] = df["processed_image_path"].apply(
    lambda x: str(DATASET_ROOT / x)
)

# Filtrar apenas imagens que existem em disco
mask = df["image_path"].apply(os.path.exists)
df = df[mask].copy()

print("\nDistribuição final de classes:")
print(df["label_4class"].value_counts())
print("\nTotal imagens disponíveis:", len(df))

Após filtro Keep: (15216, 12)

Distribuição final de classes:
label_4class
Lithiasis        726
Stricture        392
Normal           299
Biliary Leaks    151
Name: count, dtype: int64

Total imagens disponíveis: 1568


In [41]:
# ============================================================
# CELL 5 — SPLIT POR PACIENTE (sem data leakage)
# ============================================================

# Split por patient_id para evitar data leakage entre treino e teste
patient_col = "patient_id"
patients = df[patient_col].unique()

train_patients, temp_patients = train_test_split(
    patients, test_size=0.30, random_state=SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=0.50, random_state=SEED
)

df_train = df[df[patient_col].isin(train_patients)].copy()
df_val   = df[df[patient_col].isin(val_patients)].copy()
df_test  = df[df[patient_col].isin(test_patients)].copy()

print(f"Train: {len(df_train)} imagens ({df_train[patient_col].nunique()} pacientes)")
print(f"Val:   {len(df_val)} imagens ({df_val[patient_col].nunique()} pacientes)")
print(f"Test:  {len(df_test)} imagens ({df_test[patient_col].nunique()} pacientes)")

print("\nDistribuição no treino:")
print(df_train["label_4class"].value_counts())
print("\nDistribuição na validação:")
print(df_val["label_4class"].value_counts())
print("\nDistribuição no teste:")
print(df_test["label_4class"].value_counts())

Train: 1166 imagens (305 pacientes)
Val:   229 imagens (65 pacientes)
Test:  173 imagens (66 pacientes)

Distribuição no treino:
label_4class
Lithiasis        533
Stricture        300
Normal           218
Biliary Leaks    115
Name: count, dtype: int64

Distribuição na validação:
label_4class
Lithiasis        94
Stricture        58
Normal           48
Biliary Leaks    29
Name: count, dtype: int64

Distribuição no teste:
label_4class
Lithiasis        99
Stricture        34
Normal           33
Biliary Leaks     7
Name: count, dtype: int64


In [42]:
# ============================================================
# CELL 6 — TRANSFORMS (com augmentation agressivo para treino)
# ============================================================

# Estatísticas do ImageNet — adequadas para modelos pré-treinados
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    # Simulação de variações de contraste comuns em fluoroscopia
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.4,
        saturation=0.1,
        hue=0.05
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    # Apaga patches aleatórios (regularização)
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms definidos.")

Transforms definidos.


In [43]:
# ============================================================
# CELL 7 — DATASET CLASS
# ============================================================

class CPREDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["target"])
        return img, label

print("Dataset class definida.")

Dataset class definida.


In [44]:
# ============================================================
# CELL 8 — DATALOADERS COM WEIGHTED SAMPLER (oversampling)
# ============================================================

train_dataset = CPREDataset(df_train, transform=train_transforms)
val_dataset   = CPREDataset(df_val,   transform=val_test_transforms)
test_dataset  = CPREDataset(df_test,  transform=val_test_transforms)

# Weighted Random Sampler: oversampling das classes minoritárias
train_labels = df_train["target"].values
class_counts = np.bincount(train_labels)
class_weights_sampler = 1.0 / class_counts
sample_weights = class_weights_sampler[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 37
Val batches:   8
Test batches:  6


In [45]:
# ============================================================
# CELL 9 — MODELO: EfficientNet-B3 com Fine-tuning
# ============================================================
# Razão da escolha:
#   - EfficientNet-B3 tem melhor relação parâmetros/performance que ResNet18
#   - Compound scaling (depth + width + resolution) é especialmente útil
#     para imagens médicas com features subtis
#   - Pré-treinado em ImageNet → excelente ponto de partida mesmo para raios-X
# ============================================================

gc.collect()
torch.cuda.empty_cache()

model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)

# Descongelar todas as camadas para fine-tuning completo
for param in model.parameters():
    param.requires_grad = True

# Substituir o classificador final
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, len(class_names))
)

model = model.to(device)
print("Classificador:", model.classifier)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis: {total_params:,}")

Classificador: Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1536, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=256, out_features=4, bias=True)
)
Parâmetros treináveis: 11,090,732


In [46]:
# ============================================================
# CELL 10 — LOSS COM CLASS WEIGHTS + OPTIMIZADOR + SCHEDULER
# ============================================================

# Class weights para a loss function (complementa o sampler)
# Garante que erros nas classes raras têm mais peso no gradiente
class_weights_loss = torch.tensor(
    [1.0 / c for c in class_counts],
    dtype=torch.float32
)
class_weights_loss = class_weights_loss / class_weights_loss.sum() * len(class_names)
class_weights_loss = class_weights_loss.to(device)

print("Class weights (loss):")
for i, (name, w) in enumerate(zip(class_names, class_weights_loss.cpu())):
    print(f"  {name}: {w:.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights_loss)

# Optimizador com learning rates diferenciados:
# backbone (camadas pré-treinadas) → LR muito baixo
# classifier (camadas novas) → LR normal
optimizer = optim.AdamW([
    {"params": model.features.parameters(), "lr": LR_BACKBONE},
    {"params": model.classifier.parameters(), "lr": LR},
], weight_decay=WEIGHT_DECAY)

# Cosine Annealing: LR vai diminuindo suavemente ao longo das epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-7
)

print("\nOptimizador e scheduler configurados.")

Class weights (loss):
  Biliary Leaks: 1.8809
  Lithiasis: 0.4058
  Normal: 0.9922
  Stricture: 0.7210

Optimizador e scheduler configurados.


In [47]:
# ============================================================
# CELL 11 — FUNÇÃO DE AVALIAÇÃO
# ============================================================

def evaluate(model, loader, return_probs=False):
    model.eval()
    loss_total = 0.0
    y_true, y_pred, y_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_total += loss.item()
            probs = F.softmax(outputs, dim=1)
            preds = probs.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    metrics = {
        "loss":              loss_total / len(loader),
        "accuracy":          accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if return_probs:
        return metrics, y_true, y_pred, y_probs
    return metrics, y_true, y_pred

print("Função evaluate definida.")

Função evaluate definida.


In [48]:
# ============================================================
# CELL 12 — TREINO COM EARLY STOPPING
# ============================================================

best_f1       = 0.0
patience      = 7       # epochs sem melhoria antes de parar
patience_ctr  = 0
history       = []

print(f"A treinar por até {EPOCHS} epochs (early stopping patience={patience})...\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    t0 = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient clipping — estabiliza o treino
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    train_loss = running_loss / len(train_loader)
    val_metrics, _, _ = evaluate(model, val_loader)
    elapsed = time.time() - t0

    history.append({
        "epoch":      epoch + 1,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val F1: {val_metrics['macro_f1']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"{elapsed:.0f}s"
    )

    if val_metrics["macro_f1"] > best_f1:
        best_f1 = val_metrics["macro_f1"]
        patience_ctr = 0
        torch.save(
            model.state_dict(),
            MODELS_DIR / "best_efficientnet_b3.pt"
        )
        print(f"  ✓ Melhor modelo guardado (F1={best_f1:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping na epoch {epoch+1}.")
            break

print(f"\nTreino concluído. Melhor Val F1: {best_f1:.4f}")

A treinar por até 30 epochs (early stopping patience=7)...



OutOfMemoryError: CUDA out of memory. Tried to allocate 62.00 MiB. GPU 0 has a total capacity of 47.37 GiB of which 34.69 MiB is free. Process 39870 has 614.00 MiB memory in use. Process 66684 has 904.00 MiB memory in use. Process 69366 has 632.00 MiB memory in use. Process 91425 has 692.00 MiB memory in use. Process 101875 has 644.00 MiB memory in use. Process 167335 has 578.00 MiB memory in use. Process 214097 has 670.00 MiB memory in use. Process 273533 has 820.00 MiB memory in use. Process 273876 has 930.00 MiB memory in use. Process 273893 has 860.00 MiB memory in use. Process 273859 has 1.27 GiB memory in use. Process 291730 has 514.00 MiB memory in use. Process 293805 has 902.00 MiB memory in use. Process 300762 has 922.00 MiB memory in use. Including non-PyTorch memory, this process has 1.48 GiB memory in use. Of the allocated memory 967.58 MiB is allocated by PyTorch, and 46.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting 

In [ ]:
# ============================================================
# CELL 13 — CURVAS DE TREINO
# ============================================================

history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
axes[0].plot(history_df["epoch"], history_df["loss"], label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["macro_f1"])
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1")
axes[1].set_title("Validation Macro F1")
axes[1].axhline(y=0.738, color="r", linestyle="--", label="Baseline 0.738")
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["accuracy"])
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Accuracy")
axes[2].set_title("Validation Accuracy")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 14 — AVALIAÇÃO FINAL NO TESTE
# ============================================================

# Carregar o melhor modelo guardado
model.load_state_dict(
    torch.load(MODELS_DIR / "best_efficientnet_b3.pt", map_location=device)
)

test_metrics, y_true, y_pred, y_probs = evaluate(
    model, test_loader, return_probs=True
)

print("=" * 50)
print("RESULTADOS NO CONJUNTO DE TESTE")
print("=" * 50)
print(f"  Accuracy:          {test_metrics['accuracy']:.4f}")
print(f"  Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
print(f"  Macro F1-Score:    {test_metrics['macro_f1']:.4f}")
print(f"  Baseline (artigo): 0.7380")
print(f"  Diferença:         {test_metrics['macro_f1'] - 0.738:+.4f}")
print("=" * 50)

# AUC-ROC multi-classe (One vs Rest)
y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
y_probs_arr = np.array(y_probs)
auc_macro = roc_auc_score(y_true_bin, y_probs_arr, average="macro", multi_class="ovr")
print(f"  AUC-ROC (macro):   {auc_macro:.4f}")

In [ ]:
# ============================================================
# CELL 15 — CLASSIFICATION REPORT
# ============================================================

print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    zero_division=0
))

In [ ]:
# ============================================================
# CELL 16 — MATRIZ DE CONFUSÃO
# ============================================================

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title in zip(
    axes,
    [cm, cm_norm],
    ["Confusion Matrix (counts)", "Confusion Matrix (normalised)"]
):
    im = ax.imshow(data, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = f"{data[i,j]:.2f}" if data.dtype == float else str(data[i,j])
            color = "white" if data[i,j] > data.max() * 0.5 else "black"
            ax.text(j, i, val, ha="center", va="center", color=color, fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 17 — GRAD-CAM (Interpretabilidade) — OBRIGATÓRIO
# ============================================================
# Grad-CAM: usa os gradientes da última camada convolucional
# para gerar um mapa de calor das regiões mais importantes
# para a decisão do modelo.
# ============================================================

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        # Pooling global dos gradientes
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()

        # Normalizar entre 0 e 1
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam, class_idx


def show_gradcam(model, dataset, indices, gradcam, save_path=None):
    """
    Mostra imagem original + heatmap Grad-CAM para os índices pedidos.
    """
    n = len(indices)
    fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
    if n == 1:
        axes = [axes]

    denorm = transforms.Normalize(
        mean=[-m/s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
        std=[1/s for s in IMAGENET_STD]
    )

    for row, idx in enumerate(indices):
        img_tensor, true_label = dataset[idx]
        inp = img_tensor.unsqueeze(0).to(device)
        inp.requires_grad_(False)

        cam, pred_class = gradcam.generate(inp)

        # Imagem original desnormalizada
        img_vis = denorm(img_tensor).permute(1, 2, 0).clamp(0, 1).numpy()

        # Redimensionar CAM para o tamanho da imagem
        import cv2
        cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        heatmap = cm.jet(cam_resized)[:, :, :3]
        overlay = 0.55 * img_vis + 0.45 * heatmap
        overlay = np.clip(overlay, 0, 1)

        axes[row][0].imshow(img_vis, cmap="gray")
        axes[row][0].set_title(
            f"Original\nReal: {idx_to_class[true_label]}",
            fontsize=10
        )
        axes[row][0].axis("off")

        axes[row][1].imshow(overlay)
        axes[row][1].set_title(
            f"Grad-CAM\nPredito: {idx_to_class[pred_class]}",
            fontsize=10
        )
        axes[row][1].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


# Camada alvo: última camada convolucional do EfficientNet-B3
target_layer = model.features[-1][0]  # último bloco MBConv
gradcam = GradCAM(model, target_layer)

# Mostrar 1 exemplo por classe
sample_indices = []
for cls_idx in range(len(class_names)):
    candidates = [i for i, (_, lbl) in enumerate(test_dataset) if lbl == cls_idx]
    if candidates:
        sample_indices.append(random.choice(candidates))

print(f"A gerar Grad-CAM para {len(sample_indices)} exemplos (1 por classe)...")
show_gradcam(
    model, test_dataset, sample_indices, gradcam,
    save_path=RESULTS_DIR / "gradcam_examples.png"
)
print("Grad-CAM guardado em", RESULTS_DIR / "gradcam_examples.png")

In [ ]:
# ============================================================
# CELL 18 — COMPARAÇÃO FINAL COM BASELINE
# ============================================================

baseline_f1 = 0.738

print("=" * 50)
print("COMPARAÇÃO COM BASELINE DO ARTIGO")
print("=" * 50)
print(f"  Baseline Macro F1: {baseline_f1:.4f}")
print(f"  Nosso Macro F1:    {test_metrics['macro_f1']:.4f}")
diff = test_metrics['macro_f1'] - baseline_f1
symbol = "✓ SUPERA" if diff > 0 else "✗ ABAIXO"
print(f"  Diferença:         {diff:+.4f}  → {symbol} a baseline")
print("=" * 50)

# Guardar métricas em CSV para o relatório
pd.DataFrame([{
    "model": "EfficientNet-B3",
    **test_metrics,
    "auc_roc_macro": auc_macro,
    "baseline_f1": baseline_f1,
    "delta_f1": diff
}]).to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
print("Métricas guardadas em", RESULTS_DIR / "test_metrics.csv")